# Train on Kaggle

Runs `mlops-project` on a Kaggle GPU with the same command as on the laptop, then zips the run folder so it can go home.

**Before running**

1. On the laptop, in `mlops-project/`: `uv run python ../kaggle-bridge/pack.py` → `kaggle-bridge/upload-to-kaggle/code.zip` and `data.zip`.
2. Kaggle → Datasets → New Dataset: upload `code.zip` (e.g. `mlops-project-code`) and `data.zip` (e.g. `mlops-project-data`), both **Private**.
3. This notebook → **Settings**: Accelerator **GPU T4 x2** (not P100), Internet **On**.
4. **Add Input**: both datasets.

**Running**: run cells 1–4 one by one to check, then **Save Version → Save & Run All (Commit)**. It keeps running with the browser closed.

**After**: Output tab → download `<run_name>.zip` → in `mlops-project/` on the laptop: `uv run python ../kaggle-bridge/unpack.py`, then `uv run python -m pipelines.log_run_pipeline`.

The run name comes from `configs/train.yaml` inside `code.zip`.

## 1. Settings (the only cell to edit)

In [ ]:
USE_UV = True  # False = install requirements.txt with pip into Kaggle's own Python
CODE_INPUT = None  # None = find .../mlops-project/ in /kaggle/input automatically
DATA_INPUT = None  # None = find .../splits/ in /kaggle/input automatically

## 2. Copy code and data to a writable folder

`/kaggle/input/` is read-only, so everything is copied to `/kaggle/working/mlops-project/`.

In [ ]:
import shutil
from pathlib import Path

PROJECT = "mlops-project"
INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working") / PROJECT


def find_one(file_name: str, folder_name: str) -> Path:
    """The one folder in /kaggle/input named folder_name that holds file_name."""

    found = []
    for path in INPUT_DIR.rglob(file_name):
        if path.parent.name == folder_name:
            found.append(path.parent)
    if len(found) != 1:
        raise FileNotFoundError(
            f"Expected one {folder_name}/{file_name} in {INPUT_DIR}, found {found}. "
            "Check the Input panel, or set CODE_INPUT / DATA_INPUT in cell 1."
        )
    return found[0]


code_dir = Path(CODE_INPUT) if CODE_INPUT else find_one("pyproject.toml", PROJECT)
splits_dir = (
    Path(DATA_INPUT) if DATA_INPUT else find_one("split_manifest.csv", "splits")
)
print("code:", code_dir)
print("data:", splits_dir)

shutil.rmtree(WORK_DIR, ignore_errors=True)  # leftovers of an earlier session
shutil.copytree(code_dir, WORK_DIR)
shutil.copytree(splits_dir, WORK_DIR / "data" / "splits")

%cd {WORK_DIR}
!ls

## 3. Install

`uv sync` installs the exact versions from `uv.lock` (on Linux, PyPI torch is the CUDA build). Takes a few minutes.

In [ ]:
if USE_UV:
    !pip install -q uv
    !uv sync
    RUN = "uv run python"
else:
    !pip install -q -r requirements.txt
    RUN = "python"

## 4. Check the GPU (stops here if there is none)

In [ ]:
gpu = !{RUN} -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"
print(gpu[-1])
if not gpu[-1].startswith("True"):
    raise RuntimeError(
        "No GPU. Settings → Accelerator: GPU T4 x2, then restart the session."
    )

## 5. Train and evaluate

`full_pipeline` notices it runs on Kaggle and stops after evaluation (MLflow is logged on the laptop).

In [ ]:
!{RUN} -m pipelines.full_pipeline

## 6. Zip the run folder

Only `<run_name>.zip` is left in the Output tab; the working copy is deleted.

In [ ]:
runs_dir = WORK_DIR / "outputs" / "runs"
runs = sorted(runs_dir.iterdir()) if runs_dir.exists() else []
if len(runs) != 1:
    raise RuntimeError(
        f"Expected one run folder in {runs_dir}, found {runs}. See cell 5."
    )

run_dir = runs[0]
zip_path = shutil.make_archive(
    f"/kaggle/working/{run_dir.name}", "zip", root_dir=runs_dir, base_dir=run_dir.name
)

%cd /kaggle/working
shutil.rmtree(WORK_DIR)
print("Download from the Output tab:", zip_path)
!ls -lh /kaggle/working